## Memento

---

> **In one line.** A memento is an *opaque snapshot* $m$ of an object's state $s$: the originator wraps its current configuration with $\mathrm{save} : S \to M$ and later recovers it with $\mathrm{restore} : M \to S$, and these two maps are inverse on the round trip — $\mathrm{restore} \circ \mathrm{save} = \mathrm{id}_S$ — so saving then restoring returns *exactly* the original state, while a caretaker holds the mementos without ever looking inside them.

### 1. Two spaces and the maps between them

Let $S$ be the space of all possible **states** — every configuration the originator can be in — and let $s \in S$ be one specific state, the originator's current configuration at a single moment in time. Mirroring it is $M$, the space of all **mementos**: opaque objects, each of which captures one state. A particular memento $m \in M$ is a snapshot of one $s$, and crucially its internals are visible only to the originator.

The pattern is two functions, one in each direction. The **snapshot** map

$$\mathrm{save} : S \longrightarrow M$$

takes the current state and wraps it in a memento. Running the other way, the **recovery** map

$$\mathrm{restore} : M \longrightarrow S$$

unwraps a memento and returns the state it carried. Read together they form an out-and-back: a state leaves $S$, lives for a while as an opaque blob in $M$, and is later pulled back into $S$ unchanged.

### 2. Save and restore as inverse maps

The whole point is that nothing is lost on that round trip. Compose the two in the order *save, then restore*, and you must land exactly where you started:

$$\boxed{\,\mathrm{restore} \circ \mathrm{save} = \mathrm{id}_S\,}$$

where $\mathrm{id}_S$ is the identity on $S$. Pointwise, for any state $s \in S$,

$$\mathrm{restore}(\mathrm{save}(s)) = s.$$

So $\mathrm{restore}$ acts as a **left inverse** of $\mathrm{save}$: the memento is a faithful encoding, and decoding it recovers the original with no information lost. Following one state through the save-then-restore cycle makes the data flow explicit:

$$\underbrace{s}_{S} \;\xrightarrow{\;\mathrm{save}\;}\; \underbrace{m}_{M} \;\xrightarrow{\;\mathrm{restore}\;}\; \underbrace{s}_{S} \qquad\Longrightarrow\qquad \mathrm{restore}\big(\mathrm{save}(s)\big) = s \;\;\checkmark$$

The originator owns both arrows: it is the only object that can build a memento from its state and the only one that can read one back. A **caretaker** sits alongside, holding an ordered history $[m_1, m_2, \ldots, m_n]$ of saved snapshots, but it treats each $m_i$ as a sealed envelope — it stores and returns them, never reads them.

### 3. Conditions

1. **Round-trip identity** — $\mathrm{restore}(\mathrm{save}(s)) = s$ for every $s \in S$, equivalently $\mathrm{restore} \circ \mathrm{save} = \mathrm{id}_S$. Saving then restoring gives back exactly the original state, with no information lost.
2. **Encapsulation** — every $m \in M$ is opaque to all except the originator. The caretaker stores mementos but never reads or modifies their contents; only the originator may inspect a memento's internals.
3. **History enables time-travel** — a stored list $[m_1, \ldots, m_n]$ lets the originator jump to *any* prior state $s_i = \mathrm{restore}(m_i)$, not merely the most recent one. The caretaker's history turns single-step undo into arbitrary rewind.

&nbsp;

> 💾 A game save file. $\mathrm{save}(s) = m$ captures your state; $\mathrm{restore}(m) = s$ reloads it. The save file is an opaque blob ($m \in M$) — only the game (originator) can read what's inside it. The save menu (caretaker) just keeps the files in a list and hands them back when asked.

### Exercise 17 — Text Editor Snapshots

---

**Scenario:** An editor (originator) can save snapshots and restore to any of them — not just the most recent. A `History` (caretaker) stores $[m_1, m_2, \ldots]$ but never reads them.

**Your task:** Implement `Editor`, `EditorMemento`, and `History`. Verify the round-trip identity $\mathrm{restore}(\mathrm{save}(s)) = s$.

```python
editor = Editor()
history = History()                  # caretaker
editor.write("Hello")
history.save(editor.save())          # save(s) = m_1
editor.write(" World")
editor.restore(history.undo())       # restore(m_1) = s_1
print(editor.text)                   # Hello — round-trip identity holds
```

**Hints**

- `editor.save()` is $\mathrm{save}(s)$ — returns an `EditorMemento(self._text)`. `editor.restore(m)` is $\mathrm{restore}(m)$ — sets `self._text = m.get_state()`.
- The caretaker (`History`) holds `self._stack = []` and never calls `m.get_state()` — it only stores and returns mementos. This enforces the encapsulation condition.

In [ ]:
# --------------------------------
# Memento: m in M — an opaque snapshot of one state s

class EditorMemento:
    def __init__(self, text):
        self._text = text                    # the captured state s

    def get_state(self):                     # only the originator reads this
        return self._text

# --------------------------------
# Originator: the object whose state s is snapshotted

class Editor:
    def __init__(self):
        self._text = ""                      # current state s

    @property
    def text(self):
        return self._text

    def write(self, more):
        self._text += more                   # mutate state s

    def save(self):                          # save(s) = m: wrap current state
        ...

    def restore(self, memento):              # restore(m) = s: unwrap into state
        ...

# --------------------------------
# Caretaker: stores [m_1, m_2, ...] but never reads their contents

class History:
    def __init__(self):
        self._stack = []

    def save(self, memento):                 # store m — do NOT call get_state()
        ...

    def undo(self):                          # return a stored m
        ...

# --------------------------------
editor = Editor()
history = History()                  # caretaker
editor.write("Hello")
history.save(editor.save())          # save(s) = m_1
editor.write(" World")
editor.restore(history.undo())       # restore(m_1) = s_1
print(editor.text)                   # expect: Hello — round-trip identity holds

### Exercise 18 — Game Save System

---

**Scenario:** A game character (originator) has health, position, and inventory (together forming $s \in S$). The player saves at named checkpoints. The round-trip identity must hold for all three attributes simultaneously.

**Your task:** Build a named checkpoint system: `save("boss")` stores $m$; `load("boss")` restores $s$.

```python
game = Game()
saves = SaveManager()                  # caretaker
game.health = 50
game.inventory.append("sword")
saves.save("boss", game.save())        # save(s) = m
game.health = 10                       # take damage
game.restore(saves.load("boss"))       # restore(m) = s
print(game.health, game.inventory)     # 50 ['sword'] — round-trip identity holds
```

**Hints**

- The caretaker stores `self._saves = {"boss": m}`. The memento must capture the full $s$ as a deep copy — partial snapshots break $\mathrm{restore} \circ \mathrm{save} = \mathrm{id}_S$.
- `game.save()` is $\mathrm{save}(s)$ — return a `GameMemento` holding a deep copy of health, position, and inventory. `game.restore(m)` is $\mathrm{restore}(m)$ — reload all three from the memento.

In [ ]:
import copy

# --------------------------------
# Memento: m in M — captures the FULL state s (deep copy)

class GameMemento:
    def __init__(self, health, position, inventory):
        # deep copy so later mutation of s does not alter m
        self._state = copy.deepcopy((health, position, inventory))

    def get_state(self):                     # only the originator reads this
        return self._state

# --------------------------------
# Originator: state s = (health, position, inventory)

class Game:
    def __init__(self):
        self.health = 100
        self.position = (0, 0)
        self.inventory = []

    def save(self):                          # save(s) = m: snapshot full state
        ...

    def restore(self, memento):              # restore(m) = s: reload all three
        ...

# --------------------------------
# Caretaker: named checkpoints {name: m} — never reads memento contents

class SaveManager:
    def __init__(self):
        self._saves = {}

    def save(self, name, memento):           # store m under a name
        ...

    def load(self, name):                    # return the stored m
        ...

# --------------------------------
game = Game()
saves = SaveManager()                # caretaker
game.health = 50
game.inventory.append("sword")
saves.save("boss", game.save())      # save(s) = m
game.health = 10                     # take damage
game.restore(saves.load("boss"))     # restore(m) = s
print(game.health, game.inventory)   # expect: 50 ['sword'] — round-trip identity holds